# Intro

- Recoding of https://github.com/Daniel-EST/deep-steganography/ but updated to match modern tensorflow or pytorch depending what is easiest to use
- from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels
    - with rbg colour (no greyscale) to ease training

- overall process:
  - get text which is small enough *after* huffman encoding to fit within 64*64*3 sized block, and encode within image using LSB resulting in **secret image** which will be used by NN (for training all will be encoded with *"Hello World!"* just to keep things simple, but randomized text could be more authentic)
    - LSB - use pillow library (https://pypi.org/project/pillow/) to ease implementation
    - Huffman will also be from GeeksForGeeks / public implementation to speed development
  - NN will be three layers a preperation > hide > reveal networks (with each being NNs) that will ideally allow for full reconstruction of secret image
  - finally undo LSB + Huffman to retrieve final text and check accordingly.

- Issues so far:
  - Deep stenography is greatly outdated so fixing has not been very easy, could promp re-scope of assign to meet deadline if it is unable to be done
  - reformatting between secret image > steg image > revealed secret causes issues with LSB where unless its 100% (>90% ssim) then final classical steganography cannot be performed often
  - could add scheduler to improve flattening learning rate after x epochs and this could benefit the issues with too much noise ruining secret text recovery.

In [ ]:
from LSB_Steganography.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

import tensorflow as tf
# print(tf.config.list_physical_devices())
# x = tf.constant([1.0, 2.0])
# print(f"Tensor device: {x.device}")

In [ ]:
from datasets import load_dataset
import os, certifi
import numpy as np
import math

os.environ['SSL_CERT_FILE'] = certifi.where()
def is_rgb(example):
    return example['image'].mode == 'RGB'
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

# comment out select for whole dataset, just to test set to 1/4
num_samples = 25000 #len(tiny_imageNet_rgb['train'])
train_pool = tiny_imageNet_rgb['train'].select(range(num_samples))
test_pool = tiny_imageNet_rgb['valid']

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# 2. Sample from Training Pool
def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def create_tfrecord(dataset_pool, output_path):
    with tf.io.TFRecordWriter(output_path) as writer:
        for i in range(len(dataset_pool)):
            # Pair cover (i) and secret (i+1) as per your logic
            cover_idx = i
            secret_idx = (i + 1) % len(dataset_pool)
            
            # Process cover image
            c_img = dataset_pool[cover_idx]['image'].convert('RGB').resize((64, 64))
            
            # Process secret image with LSB
            s_img_raw = dataset_pool[secret_idx]['image'].convert('RGB').resize((64, 64))
            s_steg = LSBEmbed(encoded_data, np.array(s_img_raw)) # Using your LSB function
            
            # Serialize
            feature = {
                'cover': _bytes_feature(np.array(c_img).tobytes()),
                'secret': _bytes_feature(np.array(s_steg).tobytes())
            }
            example = tf.train.Example(features=tf.train.Features(feature=feature))
            writer.write(example.SerializeToString())

# Run for train and test
create_tfrecord(train_pool, 'train_data.tfrecords')
create_tfrecord(test_pool, 'test_data.tfrecords')

# 1. Prepare your message mask once (outside the function)
# We convert the binary data into a bit-mask tensor of 0s and 1s
def create_binary_mask(encoded_data, target_shape=(64, 64, 3)):
    # Convert bit-string/bytes to a numpy array of bits
    # This is a placeholder logic: in practice, you'd flatten the bits 
    # and pad them to match target_shape.size
    total_bits = np.prod(target_shape)
    # Just an example: repeating the encoded data bits to fill the mask
    bit_array = np.unpackbits(np.frombuffer(encoded_data, dtype=np.uint8))
    mask = np.resize(bit_array, total_bits).reshape(target_shape)
    return tf.constant(mask, dtype=tf.uint8)

MESSAGE_MASK = create_binary_mask(encoded_data)

def parse_function(example_proto):
    feature_description = {
        'cover': tf.io.FixedLenFeature([], tf.string),
        'secret': tf.io.FixedLenFeature([], tf.string),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_description)
    
    # 1. Decode Cover
    cover = tf.io.decode_raw(parsed['cover'], tf.uint8)
    cover = tf.reshape(cover, [64, 64, 3])
    
    # 2. Decode Secret and Apply Vectorized LSB
    secret_raw = tf.io.decode_raw(parsed['secret'], tf.uint8)
    secret_raw = tf.reshape(secret_raw, [64, 64, 3])
    
    # Clear LSB (AND 254) and Inject Message (OR Mask)
    # This replaces the LSBEmbed call entirely
    secret_steg = tf.bitwise.bitwise_and(secret_raw, 254)
    secret_steg = tf.bitwise.bitwise_or(secret_steg, MESSAGE_MASK)
    
    # 3. Cast and Normalize once
    # Removing the double-division bug found in the model
    cover = tf.cast(cover, tf.float32) / 255.0
    secret = tf.cast(secret_steg, tf.float32) / 255.0
    
    return cover, secret

print(f"Pools Ready: {num_samples/2} Train pairs, {len(test_pool)/2} Test pairs.")

In [ ]:
from tensorflow.keras import layers, losses
tf.keras.backend.clear_session()
class SteganoModel(tf.keras.Model):
    def __init__(self):
        super(SteganoModel, self).__init__()
        self.beta = 1
        
        # 1. PREP NETWORK: (None, 64, 64, 3) -> (None, 64, 64, 150)
        # Using 50 filters of each size (3x3, 4x4, 5x5) to reach 150
        self.prep_1_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.prep_1_conv4 = layers.Conv2D(50, (4, 4), padding='same', activation='relu')
        self.prep_1_conv5 = layers.Conv2D(50, (5, 5), padding='same', activation='relu')
        self.prep_2_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.prep_2_conv4 = layers.Conv2D(50, (4, 4), padding='same', activation='relu')
        self.prep_2_conv5 = layers.Conv2D(50, (5, 5), padding='same', activation='relu')

        # 2. HIDE NETWORK: (None, 64, 64, 150+3) -> (None, 64, 64, 3)
        self.hide_1_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.hide_1_conv4 = layers.Conv2D(50, (4,4), padding='same', activation='relu')
        self.hide_1_conv5 = layers.Conv2D(50, (5,5), padding='same', activation='relu')
        self.hide_2_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.hide_2_conv4 = layers.Conv2D(50, (4,4), padding='same', activation='relu')
        self.hide_2_conv5 = layers.Conv2D(50, (5,5), padding='same', activation='relu')
        self.hide_3_conv1 = layers.Conv2D(3, (1,1), padding='same', activation='relu')

        # 3. REVEAL NETWORK: (None, 64, 64, 3) -> (None, 64, 64, 3)
        self.reveal_1_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.reveal_1_conv4 = layers.Conv2D(50, (4,4), padding='same', activation='relu')
        self.reveal_1_conv5 = layers.Conv2D(50, (5,5), padding='same', activation='relu')
        self.reveal_2_conv3 = layers.Conv2D(50, (3, 3), padding='same', activation='relu')
        self.reveal_2_conv4 = layers.Conv2D(50, (4,4), padding='same', activation='relu')
        self.reveal_2_conv5 = layers.Conv2D(50, (5,5), padding='same', activation='relu')
        self.reveal_3_conv1 = layers.Conv2D(3, (1,1), padding='same', activation='relu')

    def hide(self, secret, cover):
        # Prep
        p1_3 = self.prep_1_conv3(secret)
        p1_4 = self.prep_1_conv4(secret)
        p1_5 = self.prep_1_conv5(secret)
        p_concat1 = tf.concat([p1_3, p1_4, p1_5], axis=3) # 150 channels

        p2_3 = self.prep_2_conv3(p_concat1)
        p2_4 = self.prep_2_conv4(p_concat1)
        p2_5 = self.prep_2_conv5(p_concat1)
        p_concat2 = tf.concat([p2_3,p2_4,p2_5],axis=3)

        # Hide
        h_concat1 = tf.concat([p_concat2, cover], axis=3) # 153 channels
        h1_3 = self.hide_1_conv3(h_concat1)
        h1_4 = self.hide_1_conv4(h_concat1)
        h1_5 = self.hide_1_conv5(h_concat1)
        h_concat2 = tf.concat([h1_3,h1_4,h1_5], axis=3) # 153 channels
        h2_3 = self.hide_2_conv3(h_concat2)
        h2_4 = self.hide_2_conv4(h_concat2)
        h2_5 = self.hide_2_conv5(h_concat2)
        h_concat3 = tf.concat([h2_3,h2_4,h2_5], axis=3) # 153 channels
        return self.hide_3_conv1(h_concat3)

    def reveal(self, stego):
        r1_3 = self.reveal_1_conv3(stego)
        r1_4 = self.reveal_1_conv4(stego)
        r1_5 = self.reveal_1_conv5(stego)
        r_concat1 = tf.concat([r1_3,r1_4,r1_5],axis=3)

        r2_3 = self.reveal_2_conv3(r_concat1)
        r2_4 = self.reveal_2_conv4(r_concat1)
        r2_5 = self.reveal_2_conv5(r_concat1)
        r_concat2 = tf.concat([r2_3,r2_4,r2_5],axis=3)

        return self.reveal_3_conv1(r_concat2)

    def call(self, inputs):
        cover, secret = inputs
        stego = self.hide(secret, cover)
        extracted = self.reveal(stego)
        return stego, extracted

    def train_step(self, data):
        # Unpack the data (Expecting cover and secret images)
        cover, secret = data

        with tf.GradientTape() as tape:
            stego = self.hide(secret,cover)
            
            noisy_stego = tf.random.normal(shape=tf.shape(stego),mean=0.0,stddev=0.01)
            
            extracted = self.reveal(noisy_stego)
            
            # Loss Calculation: total_loss = cover_mse + beta * secret_mse
            cover_mse = tf.reduce_mean(losses.mse(cover, stego))
            secret_mse = tf.reduce_mean(losses.mse(secret, extracted))
            total_loss = cover_mse + (self.beta * secret_mse)

        # Apply gradients
        trainable_vars = self.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        
        return {"loss": total_loss, "cover_mse": cover_mse, "secret_mse": secret_mse}

# Initialize
model = SteganoModel()

In [ ]:
import time
# --- Training Configuration ---
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),jit_compile=True)

train_ds = tf.data.TFRecordDataset('train_data.tfrecords') \
    .map(parse_function,num_parallel_calls=tf.data.AUTOTUNE) \
    .cache() \
    .shuffle(buffer_size=1000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

#samples / 2 (because 1 conver + 1 secret / batch)
start_time = time.time()
history = model.fit(
    train_ds, 
    epochs=EPOCHS,
    steps_per_epoch=(math.floor(num_samples/BATCH_SIZE)),
    batch_size=BATCH_SIZE
)
end_time = time.time()
total_time = end_time - start_time
print("Training Complete.")
hours = int(total_time // 3600)
minutes = int((total_time % 3600) // 60)
seconds = int(total_time % 60)

epoch_hours = int(total_time/EPOCHS // 3600)
epoch_minutes = int((total_time/EPOCHS % 3600) // 60)
epoch_seconds = int(total_time/EPOCHS % 60)

print(f"Total training time: {hours}h {minutes}m {seconds}s, time/epoch: {epoch_hours}h {epoch_minutes}m {epoch_seconds}s")

In [ ]:
# Save the whole model weights
model.save('steganography_model.keras')

# # so that reveal is seperate?
# # 1. Define a fresh input for the standalone decoder
# decoder_input = tf.keras.Input(shape=(64, 64, 3), name='stego_input_only')

# # 2. Extract the reveal logic from your trained model
# # We pass the new input through the reveal_layer of the existing 'model'
# decoder_output = model.reveal(decoder_input)

# # 3. Create the separate model
# reveal_model = tf.keras.models.Model(inputs=decoder_input, outputs=decoder_output)

# # 4. Compile and Save
# reveal_model.compile(optimizer='adam', loss='mse')
# reveal_model.save('standalone_reveal.keras')

In [ ]:
import matplotlib.pyplot as plt

def calculate_metrics(img1, img2):
    # Ensure inputs are tensors
    img1 = tf.convert_to_tensor(img1)
    img2 = tf.convert_to_tensor(img2)
    
    # If the images don't have a batch dimension (shape H,W,C), add one
    if len(img1.shape) == 3:
        img1 = tf.expand_dims(img1, axis=0)
    if len(img2.shape) == 3:
        img2 = tf.expand_dims(img2, axis=0)

    # Calculate metrics
    psnr = tf.image.psnr(img1, img2, max_val=1.0)
    ssim = tf.image.ssim(img1, img2, max_val=1.0)
    
    # .numpy() now returns an array, so [0] will work correctly
    return psnr.numpy()[0], ssim.numpy()[0]

def AccTxt(original_txt,recovered_txt):
    T = len(original_txt)

    D = 0

    for i in original_txt:
        if original_txt[i] == recovered_txt[i]:
            D+=1
    
    return D/T*100

def vectorized_lsb_extract(steg_image_tensor):
    """
    steg_image_tensor: (H, W, 3) uint8 tensor (0-255)
    Returns: (H, W, 3) uint8 tensor containing only 0s and 1s
    """
    # Bitwise AND with 1 extracts the last bit
    extracted_bits = tf.bitwise.bitwise_and(steg_image_tensor, 1)
    return extracted_bits

test_ds = tf.data.TFRecordDataset('test_data.tfrecords') \
    .map(parse_function,num_parallel_calls=tf.data.AUTOTUNE) \
        .shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for cover, secret in test_ds.batch(10).take(1):
    stego, revealed = model.predict([cover,secret])

    final_acc = 0.0
    try:
        # Convert model output [0, 1] float back to [0, 255] uint8
        revealed_uint8 = (revealed[idx] * 255).clip(0, 255).astype(np.uint8)
        
        # 2. Extract bits using the vectorized function
        bits = vectorized_lsb_extract(tf.convert_to_tensor(revealed_uint8))
        
        # 3. Pack bits into bytes
        extracted_bin = bits_to_bytes(bits)

        # 4. Decode Huffman
        final_text = HuffmanDecode(codec, extracted_bin)
        
        # Optional: Truncate to original text length to ignore Huffman padding
        final_text = final_text[:len(ORIGINAL_TEXT)]
        
        print(f"Recovered text: {final_text}")
        final_acc = AccTxt(ORIGINAL_TEXT, final_text)
    except Exception as e:
        print(f"Extraction failed due to: {e}")

    # 3. Print quantitative metrics for context
    psnr_val, ssim_val = calculate_metrics(secret[0],revealed[0])

    print(f"Metrics for Secret vs Revealed:")
    print(f"PSNR: {psnr_val:.2f} dB")
    print(f"SSIM: {ssim_val:.4f}")
    print(f"AccTxt: {final_acc:.4f}")
    plt.figure(figsize=(12, 6))

    # Original Secret Image
    plt.subplot(1, 2, 1)
    plt.title("Original Secret Image")
    plt.imshow(secret[0])
    plt.axis('off')

    # Revealed Secret Image
    plt.subplot(1, 2, 2)
    plt.title("Revealed Secret Image")
    plt.imshow(revealed[0])
    plt.axis('off')

    plt.tight_layout()
    plt.show()